# FlyLM · P100 · MaleCNS 신경 동역학과 내부 기억

전체 `Traced` 뉴런과 그 사이의 공개 연결을 보존하는 byte 언어모델.
입력은 실제 sensory annotation, 출력은 실제 motor annotation을 따른다.
현재 문자 → 작은 감각 변환기 → MaleCNS 회로 → 작은 운동 출력기 → 다음 문자.
질문 재주입·문맥 pooling·외부 언어모델·새로운 뉴런 간 연결은 없다.

뉴런 상태는 막 상태 `v`, 비음수 시냅스 신호 `s`, 느린 적응 `a` 세 값이다.
회로의 시간적 문맥은 이 상태에만 저장한다. 생물학적 기억 재현이나 대화 성능은 아직 검증되지 않았다.
뉴런 유형별 시간 상수/역치/적응 강도를 학습하지만 초기값·범위는 공학적 가정이다.
시간 단위는 simulation tick이며 ms에 대응한다고 주장하지 않는다.
역전파/Adam을 사용한다. 도파민 가소성·개별 수용체·가지별 구획은 이번 구현에 포함하지 않는다.

[MaleCNS 공식 데이터](https://male-cns.janelia.org/download/) ·
[동역학과 과제 최적화를 결합한 선행 연구](https://www.nature.com/articles/s41586-024-07939-3)

**Kaggle P100 GPU에서 위에서 아래로 실행. dynamics_t4의 가중치 구조와 호환된다. 학습 셀은 기본적으로 새 학습을 시작한다.**
기본 1,000 batch update는 실행 확인용이며 학습 완료 기준이 아니다. 대조군 실험은 포함하지 않는다.

**P100 실행 순서:** Kaggle GPU 가속기와 인터넷을 켜고 첫 코드 셀부터 실행한다.
별도 Python 프로세스로 설치된 PyTorch의 GPU 지원 목록을 확인한다.
P100에서 `sm_60` 지원이 없는 경우에만 공식 PyTorch 2.10.0 CUDA 12.6 빌드로 교체한다.
설치 후 지원 목록을 다시 확인하며, 이미 지원되는 빌드는 재설치하지 않는다.
현재 커널이 이전의 호환되지 않는 torch를 불러온 경우에만 커널을 한 번 재시작한다.
이후 실제 FP32 CUDA CSR, 작은 그래프 backward, 전체 그래프 자동 배치 탐색을 수행한다.
[공식 설치 명령](https://pytorch.org/get-started/previous-versions/),
[공식 GPU 지원표](https://github.com/pytorch/pytorch/blob/main/RELEASE.md).

**실제 다중 배치:** batch 64로 고정한다. 64-token forward/backward/Adam의 peak VRAM이
전체 VRAM의 90% 예산 안에 들어가는지 한 번 검사한다.
대화마다 독립 상태를 유지하고 padding은 상태와 loss에서 제외한다.
짧은 마지막 배치/문장은 VRAM 사용량이 내려갈 수 있다. 메모리 사용량을 위한 가짜 할당은 하지 않는다.
JSON 입력: `/kaggle/input/datasets/chika1472/dailytalk` (하위 폴더 포함).
출력: `/kaggle/working/flylm_runs/<실행시각>`.

## 1. P100 설정과 자동 배치

전체 그래프 / **batch 64** / TBPTT 64 / 문자당 4 simulation ticks / FP32.
`vram_fraction=0.90`은 다른 GPU 사용량을 제외하고 적용할 peak VRAM 예산이다.
기본 batch는 64로 고정한다. 64-token forward/backward/Adam 메모리 검사 한 번 후 학습한다. `batch_size='auto'`로 바꾸면 자동 탐색한다.
역전파·Adam 상태까지 포함해 측정한다. 탐색용 모델은 버리고 새로 학습한다.
큰 batch는 총 학습 바이트를 늘리며 기존 batch 1의 update 수와 동일한 학습량이 아니다.
`max_batch_size=None`은 데이터의 train pair 수 외에 임의 배치 상한을 두지 않는다.
Kaggle에서 인터넷을 켜야 누락된 보조 패키지와 MaleCNS 원본을 다운로드할 수 있다.

In [2]:
# Repair missing Pascal support before importing torch into this kernel.
import json
import subprocess
import sys

def disk_cuda_info():
    # A new process sees installed files, not this notebook's cached torch module.
    script = """import json, torch
print(json.dumps(dict(version=str(torch.__version__),
    capability=list(torch.cuda.get_device_capability(0)) if torch.cuda.is_available() else None,
    architectures=torch.cuda.get_arch_list())))
"""
    result = subprocess.run([sys.executable, '-c', script], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError('설치된 PyTorch 확인 실패:\n' + result.stderr)
    return json.loads(result.stdout.strip().splitlines()[-1])

disk_info = disk_cuda_info()
print('Installed PyTorch:', disk_info)
if disk_info['capability'] == [6, 0] and 'sm_60' not in disk_info['architectures']:
    print('P100 지원이 없는 빌드입니다. 공식 CUDA 12.6 빌드로 교체합니다.')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--upgrade',
        'torch==2.10.0+cu126', 'torchvision==0.25.0+cu126', 'torchaudio==2.10.0+cu126',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
    ])
    disk_info = disk_cuda_info()
    print('After installation:', disk_info)
    if 'sm_60' not in disk_info['architectures']:
        raise RuntimeError('설치 후에도 sm_60 지원이 없습니다. 위 pip 출력과 설치 정보를 확인하세요.')

loaded_torch = sys.modules.get('torch')
if (disk_info['capability'] == [6, 0] and loaded_torch is not None
        and 'sm_60' not in loaded_torch.cuda.get_arch_list()):
    raise RuntimeError(
        '디스크의 P100 지원 빌드는 확인됐지만 현재 커널에는 이전 torch가 로드돼 있습니다. '
        '커널을 한 번 재시작한 뒤 첫 셀부터 실행하세요. 다음 실행에서는 재설치하지 않습니다.'
    )
# End Pascal bootstrap.

import importlib.util
import subprocess
import sys

required = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy',
            'pyarrow': 'pyarrow', 'tqdm': 'tqdm'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('CUDA PyTorch가 제공되는 Kaggle GPU 런타임을 선택하세요.')

import gc
import hashlib
import json
import math
import platform
import random
import shutil
import time
import urllib.request
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import scipy
from scipy import sparse
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

CFG = dict(seed=42, n_neurons=None, embedding_dim=32, brain_steps=4,
           batch_size=64, vram_fraction=0.90, max_batch_size=None, tbptt=64, prompt_grad_tokens=32,
           train_updates=1000, epochs=1, lr=0.001,
           checkpoint_tokens=True, nt_confidence=0.8, use_nt_signs=True,
           edge_block=262144,
           tau_initial=(2.0, 2.0, 64.0),
           tau_min=(0.5, 0.5, 8.0), tau_max=(32.0, 16.0, 512.0),
           log_every=50, eval_every=200, save_every=200,
           eval_max_pairs=4, eval_response_tokens=128)
CACHE = Path('/kaggle/working/flylm_malecns_cache')
CACHE.mkdir(parents=True, exist_ok=True)
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])

# Validate current CUDA runtime by execution, not by an exact torch version string.
print(dict(torch=str(torch.__version__), cuda_build=torch.version.cuda,
           torch_file=torch.__file__, python=sys.executable))
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU가 보이지 않습니다. Kaggle GPU 가속기 설정과 위 PyTorch 정보를 확인하세요.')
device = torch.device('cuda')
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print(dict(gpu=gpu_name, capability=capability, compiled_architectures=torch.cuda.get_arch_list()))
if 'P100' not in gpu_name:
    print('참고: 현재 GPU는 P100이 아닙니다:', gpu_name)
try:
    probe_row = torch.tensor([0, 1, 2], dtype=torch.int64, device=device)
    probe_col = torch.tensor([1, 0], dtype=torch.int64, device=device)
    probe_values = torch.tensor([2., 3.], device=device)
    probe_matrix = torch.sparse_csr_tensor(probe_row, probe_col, probe_values, size=(2, 2))
    probe_result = torch.sparse.mm(probe_matrix, torch.ones(2, 1, device=device))
    torch.testing.assert_close(probe_result, torch.tensor([[2.], [3.]], device=device))
    torch.cuda.synchronize()
except RuntimeError as error:
    raise RuntimeError(
        f'실제 CUDA CSR 실행이 실패했습니다: {error}\n'
        '위 torch/CUDA/GPU 정보와 원본 오류를 확인하세요. P100 지원 빌드 확인 이후의 실제 연산 오류입니다.'
    ) from error
print('PASS: FP32 CUDA CSR on', gpu_name, 'capability', capability)
del probe_row, probe_col, probe_values, probe_matrix, probe_result
# End CUDA runtime probe.

torch.manual_seed(CFG['seed'])
torch.cuda.manual_seed_all(CFG['seed'])

VERSIONS = dict(python=platform.python_version(), torch=torch.__version__,
                numpy=np.__version__, pandas=pd.__version__, scipy=scipy.__version__, pyarrow=pa.__version__)
print(VERSIONS)
print(torch.cuda.get_device_name(0), 'GPU GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)
assert CFG['tbptt'] == 64
assert CFG['batch_size'] == 'auto' or (isinstance(CFG['batch_size'], int) and CFG['batch_size'] >= 1)
assert 0 < CFG['vram_fraction'] < 1
assert CFG['max_batch_size'] is None or CFG['max_batch_size'] >= 1

assert CFG['epochs'] >= 1 and CFG['tbptt'] >= 1
assert CFG['train_updates'] is None or CFG['train_updates'] >= 1
assert min(CFG[k] for k in ['log_every', 'eval_every', 'save_every', 'eval_max_pairs', 'eval_response_tokens']) >= 1
assert 1 <= CFG['prompt_grad_tokens'] < CFG['tbptt']

Installed PyTorch: {'version': '2.10.0+cu126', 'capability': [6, 0], 'architectures': ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']}
{'torch': '2.10.0+cu126', 'cuda_build': '12.6', 'torch_file': '/usr/local/lib/python3.12/dist-packages/torch/__init__.py', 'python': '/usr/bin/python3'}
{'gpu': 'Tesla P100-PCIE-16GB', 'capability': (6, 0), 'compiled_architectures': ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']}
PASS: FP32 CUDA CSR on Tesla P100-PCIE-16GB capability (6, 0)
{'python': '3.12.13', 'torch': '2.10.0+cu126', 'numpy': '2.0.2', 'pandas': '2.3.3', 'scipy': '1.16.3', 'pyarrow': '24.0.0'}
Tesla P100-PCIE-16GB GPU GiB: 15.887939453125


## 1A. Kaggle JSON 디렉터리

`/kaggle/input/datasets/chika1472/dailytalk` 아래 JSON을 직접 읽는다.
ZIP 복사/압축 해제와 Google Drive 마운트는 없다.
파일 상대 경로·크기·SHA-256으로 캐시를 구분한다.
결과와 캐시는 쓰기 가능한 `/kaggle/working` 아래에 저장한다.
Kaggle 세션 종료 전 실행 결과를 노트북 출력으로 보존하거나 다운로드한다.

In [3]:
DATA_CFG = dict(json_dir='/kaggle/input/datasets/chika1472/dailytalk',
                text_field='form', validation_fraction=0.05)
DATA_ROOT = Path(DATA_CFG['json_dir'])
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Kaggle에 데이터셋을 연결하세요: {DATA_ROOT}')
JSON_FILES = sorted(p for p in DATA_ROOT.rglob('*')
                    if p.is_file() and p.suffix.lower() == '.json'
                    and '__MACOSX' not in p.parts and not p.name.startswith('._'))
if not JSON_FILES:
    raise ValueError(f'JSON 파일이 없습니다: {DATA_ROOT}')
DATA_CACHE = Path('/kaggle/working/flylm_dialogue_cache')
DATA_CACHE.mkdir(parents=True, exist_ok=True)
dataset_sources = []
for path in tqdm(JSON_FILES, desc='JSON fingerprint'):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b''):
            digest.update(block)
    dataset_sources.append(dict(path=path.relative_to(DATA_ROOT).as_posix(),
                                bytes=path.stat().st_size, sha256=digest.hexdigest()))
dataset_sha256 = hashlib.sha256(json.dumps(dataset_sources, sort_keys=True).encode()).hexdigest()
print(dict(json_directory=str(DATA_ROOT), json_files=len(JSON_FILES), dataset_sha256=dataset_sha256))

JSON fingerprint:   0%|          | 0/14229 [00:00<?, ?it/s]

{'json_directory': '/kaggle/input/datasets/chika1472/dailytalk', 'json_files': 14229, 'dataset_sha256': '16d43f6c89309caa8aa89bc66df5f45e0ecd337bbb7eef5fa121b7adeeba9d86'}


## 1B. 두 화자의 입력–답변 쌍과 문서 단위 split

JSON 배열 순서를 유지한다. form의 연속 공백만 정리하고 original_form/인구통계/시간 metadata는 학습 텍스트에 넣지 않는다.
text_field를 original_form으로 바꾸면 그 필드로 다시 전처리한다. 지정 필드의 누락/잘못된 타입은 위치를 표시하고 중단한다.
speaker_id로 화자를 구분하며 동일 화자의 연속 발화를 하나로 합친다. 마지막 화자1에 답변이 없으면 해당 turn은 쓰지 않는다.

각 학습 쌍은 다음 구조다.

    BOS + "화자1: 입력 내용\n화자2: " + 답변 내용 + EOS
    └──────────── loss 제외 ───────────┘  └── loss 적용 ──┘

쌍마다 prompt_tokens 경계를 기록한다. uint16 토큰 캐시에 입력과 답변을 저장하되, prompt는 label -100으로 mask한다.
중복 대화/쌍을 제거한 후 대화 단위 95:5 분리를 seed와 hash로 고정한다.
같은 화자가 다른 문서에도 있을 수 있으므로 화자 독립 평가는 아니다.
최소 2개 유효 대화가 필요하며, 조건에 맞는 쌍이 없으면 가짜 답변을 만들지 않고 중단한다.


In [4]:
def dialogue_turns(document, location):
    utterances = document.get('utterance')
    if not isinstance(utterances, list):
        raise ValueError(f'{location}: utterance must be a list')
    speakers, turns = {}, []
    for index, utterance in enumerate(utterances):
        if not isinstance(utterance, dict):
            raise ValueError(f'{location}: utterance[{index}] must be an object')
        text = utterance.get(DATA_CFG['text_field'])
        if not isinstance(text, str):
            raise ValueError(f'{location}: utterance[{index}].{DATA_CFG["text_field"]} must be a string')
        text = ' '.join(text.split())
        if not text:
            continue
        speaker = utterance.get('speaker_id')
        if not isinstance(speaker, str) or not speaker.strip():
            return [], 'missing_speaker'
        if speaker not in speakers:
            speakers[speaker] = len(speakers) + 1
        role = speakers[speaker]
        if turns and turns[-1][0] == role:
            turns[-1][1].append(text)
        else:
            turns.append((role, [text]))
    if not turns:
        return [], 'empty'
    if len(speakers) != 2:
        return [], 'single_speaker' if len(speakers) == 1 else 'more_than_two_speakers'
    return [(role, ' '.join(parts)) for role, parts in turns], None

def encode_pair(question, answer):
    prefix = ('화자1: ' + question + '\n화자2: ').encode('utf-8')
    reply = answer.encode('utf-8')
    tokens = np.empty(1 + len(prefix) + len(reply) + 1, dtype='<u2')
    tokens[0], tokens[-1] = 256, 257
    tokens[1:1+len(prefix)] = np.frombuffer(prefix, dtype=np.uint8)
    tokens[1+len(prefix):-1] = np.frombuffer(reply, dtype=np.uint8)
    return tokens, 1 + len(prefix)

data_spec = dict(version=2, dataset_sha256=dataset_sha256, text_field=DATA_CFG['text_field'],
                 renderer='first_seen_two_speakers_adjacent_1to2_merged_turns_v1',
                 tokenizer='utf8_byte_bos256_eos257_uint16le', loss='response_and_eos_only',
                 seed=CFG['seed'], validation_fraction=DATA_CFG['validation_fraction'])
assert 0 < DATA_CFG['validation_fraction'] < 1
data_key = hashlib.sha256(json.dumps(data_spec, sort_keys=True).encode()).hexdigest()[:16]
token_path = DATA_CACHE / f'{data_key}.uint16'
index_path = DATA_CACHE / f'{data_key}.json'


def prepare_pairs():
    documents, pairs, seen_ids = [], [], {}
    seen_docs, seen_pairs, skipped = set(), set(), {}
    offset = 0
    partial = token_path.with_suffix('.partial')
    with partial.open('wb') as output:
        members = JSON_FILES
        for path in tqdm(members, desc='JSON → speaker1/speaker2 pairs'):
            member_name = path.relative_to(DATA_ROOT).as_posix()
            with path.open('r', encoding='utf-8-sig') as member:
                payload = json.load(member)
            if not isinstance(payload, dict) or not isinstance(payload.get('document'), list):
                raise ValueError(f'{member_name}: document list가 없습니다.')
            for number, document in enumerate(payload['document']):
                if not isinstance(document, dict):
                    raise ValueError(f'{member_name}: document[{number}] must be an object')
                doc_id = str(document.get('id') or f'{member_name}#{number}')
                turns, reason = dialogue_turns(document, f'{member_name}:{doc_id}')
                if reason:
                    skipped[reason] = skipped.get(reason, 0) + 1
                    continue
                doc_hash = hashlib.sha256(json.dumps(turns, ensure_ascii=False).encode()).hexdigest()
                if doc_id in seen_ids and seen_ids[doc_id] != doc_hash:
                    raise ValueError(f'Duplicate document ID with different content: {doc_id}')
                seen_ids[doc_id] = doc_hash
                if doc_hash in seen_docs:
                    skipped['duplicate_document'] = skipped.get('duplicate_document', 0) + 1
                    continue
                seen_docs.add(doc_hash)
                count = 0
                for i in range(len(turns)-1):
                    if turns[i][0] != 1 or turns[i+1][0] != 2:
                        continue
                    question, answer = turns[i][1], turns[i+1][1]
                    pair_hash = hashlib.sha256(json.dumps([question, answer], ensure_ascii=False).encode()).hexdigest()
                    if pair_hash in seen_pairs:
                        skipped['duplicate_pair'] = skipped.get('duplicate_pair', 0) + 1
                        continue
                    seen_pairs.add(pair_hash)
                    tokens, prompt_tokens = encode_pair(question, answer)
                    tokens.tofile(output)
                    pairs.append(dict(document=len(documents), sha256=pair_hash, offset=offset,
                                      length=len(tokens), prompt_tokens=prompt_tokens))
                    offset += len(tokens)
                    count += 1
                if count:
                    documents.append(dict(id=doc_id, member=member_name, sha256=doc_hash, pairs=count))
                else:
                    skipped['no_unique_pair'] = skipped.get('no_unique_pair', 0) + 1
    if len(documents) < 2:
        raise ValueError(f'유효한 2인 대화가 {len(documents)}개입니다. train/validation에 최소 2개 필요합니다. 제외: {skipped}')
    order = sorted(range(len(documents)), key=lambda i: hashlib.sha256(
        f'{CFG["seed"]}:{documents[i]["sha256"]}'.encode()).hexdigest())
    n_val = min(len(documents)-1, max(1, round(len(documents)*DATA_CFG['validation_fraction'])))
    validation = set(order[:n_val])
    for i, document in enumerate(documents):
        document['split'] = 'validation' if i in validation else 'train'
    for pair in pairs:
        pair['split'] = documents[pair['document']]['split']
    manifest = dict(spec=data_spec, documents=documents, pairs=pairs, total_tokens=offset,
                    json_files=len(members), skipped=skipped)
    partial.replace(token_path)
    index_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    return manifest

dataset_manifest = json.loads(index_path.read_text(encoding='utf-8')) if index_path.exists() else None
if (dataset_manifest is None or dataset_manifest['spec'] != data_spec or not token_path.exists()
        or token_path.stat().st_size != dataset_manifest['total_tokens']*2):
    dataset_manifest = prepare_pairs()
documents, pairs = dataset_manifest['documents'], dataset_manifest['pairs']
tokens_mm = np.memmap(token_path, mode='r', dtype='<u2')
train_ids = [i for i, row in enumerate(pairs) if row['split'] == 'train']
val_ids = [i for i, row in enumerate(pairs) if row['split'] == 'validation']
assert train_ids and val_ids
assert {pairs[i]['document'] for i in train_ids}.isdisjoint({pairs[i]['document'] for i in val_ids})
assert {pairs[i]['sha256'] for i in train_ids}.isdisjoint({pairs[i]['sha256'] for i in val_ids})
assert sum(row['length'] for row in pairs) == len(tokens_mm)
for row in pairs:
    start, end = row['offset'], row['offset'] + row['length']
    assert int(tokens_mm[start]) == 256 and int(tokens_mm[end-1]) == 257
    assert 1 < row['prompt_tokens'] < row['length']-1
data_stats = dict(json_files=dataset_manifest['json_files'], usable_documents=len(documents),
                  train_documents=len({pairs[i]['document'] for i in train_ids}),
                  validation_documents=len({pairs[i]['document'] for i in val_ids}),
                  train_pairs=len(train_ids), validation_pairs=len(val_ids),
                  train_response_tokens=sum(pairs[i]['length']-pairs[i]['prompt_tokens'] for i in train_ids),
                  validation_response_tokens=sum(pairs[i]['length']-pairs[i]['prompt_tokens'] for i in val_ids),
                  excluded=dataset_manifest['skipped'])
print(json.dumps(data_stats, ensure_ascii=False, indent=2))
RUN_DIR = Path('/kaggle/working/flylm_runs') / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR.mkdir(parents=True, exist_ok=False)
shutil.copyfile(index_path, RUN_DIR / 'dataset_manifest.json')
(RUN_DIR/'dataset_sources.json').write_text(json.dumps(dataset_sources, ensure_ascii=False, indent=2), encoding='utf-8')
print('Kaggle 결과 저장 경로:', RUN_DIR)

JSON → speaker1/speaker2 pairs:   0%|          | 0/14229 [00:00<?, ?it/s]

{
  "json_files": 14229,
  "usable_documents": 11520,
  "train_documents": 10944,
  "validation_documents": 576,
  "train_pairs": 331601,
  "validation_pairs": 17687,
  "train_response_tokens": 73720635,
  "validation_response_tokens": 3910781,
  "excluded": {
    "single_speaker": 380,
    "duplicate_pair": 749,
    "more_than_two_speakers": 2329
  }
}
Kaggle 결과 저장 경로: /kaggle/working/flylm_runs/20260908_014746_715747


## 2. 공식 원본 다운로드

MaleCNS v1.0 annotation, neuron neurotransmitter, connection Feather를 사용한다.
다운로드/길이 검사/SHA-256은 기존과 같다. 전체 annotation의 Traced neuron을 사용하고
그 뉴런들 사이에 존재하는 모든 edge/count를 보존한다. 생물학적 뉴런으로 확인되지 않은 orphan/glia/미할당 segment는 포함하지 않는다.


In [5]:
BASE_URL = 'https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome/'
SOURCES = {
    'annotations': ('body-annotations-male-cns-v1.0-minconf-0.5.feather', 14483314),
    'nt': ('body-neurotransmitters-male-cns-v1.0.feather', 43282834),
    'edges': ('connectome-weights-male-cns-v1.0-minconf-0.5.feather', 1051241946),
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()

def download_file(name, expected_size):
    target = CACHE / name
    if not target.exists() or target.stat().st_size != expected_size:
        if shutil.disk_usage(CACHE).free < expected_size + 512 * 1024**2:
            raise RuntimeError('데이터 다운로드에 필요한 디스크 공간이 부족합니다.')
        partial = target.with_suffix(target.suffix + '.part')
        with urllib.request.urlopen(BASE_URL + name, timeout=60) as response:
            with partial.open('wb') as f, tqdm(total=expected_size, unit='B', unit_scale=True, desc=name) as bar:
                while True:
                    chunk = response.read(8 * 1024**2)
                    if not chunk:
                        break
                    f.write(chunk)
                    bar.update(len(chunk))
        if partial.stat().st_size != expected_size:
            raise RuntimeError(f'파일 크기 불일치: {name}. v1.0 파일/다운로드를 확인하세요.')
        partial.replace(target)
    return target

files = {key: download_file(*spec) for key, spec in SOURCES.items()}
source_manifest = {key: dict(url=BASE_URL + p.name, bytes=p.stat().st_size, sha256=sha256_file(p))
                   for key, p in files.items()}
print({key: entry['bytes'] for key, entry in source_manifest.items()})


body-annotations-male-cns-v1.0-minconf-0.5.feather:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

body-neurotransmitters-male-cns-v1.0.feather:   0%|          | 0.00/43.3M [00:00<?, ?B/s]

connectome-weights-male-cns-v1.0-minconf-0.5.feather:   0%|          | 0.00/1.05G [00:00<?, ?B/s]

{'annotations': 14483314, 'nt': 43282834, 'edges': 1051241946}


## 3. 전체 뉴런, 전체 내부 연결의 CSR 캐시

Feather를 두 번 순회한다. 첫 순회는 destination별 edge 개수를 세고,
두 번째 순회는 디스크 memmap의 CSR 위치에 source/count를 채운다.
전체 원본 table을 pandas로 올리거나 edge 목록을 Python list에 누적하지 않는다.
중복 neuron pair는 count를 합산한다. CSR row=postsynaptic, column=presynaptic이다.
어떤 component도 버리지 않으며 self-edge도 원본에 있으면 유지한다.


In [7]:
def require_columns(names, required, label):
    absent = set(required)-set(names)
    if absent:
        raise ValueError(f'{label}: missing {sorted(absent)}; found {list(names)}')

def map_ids(sorted_ids, values):
    positions = np.searchsorted(sorted_ids, values)
    safe = positions.clip(0, len(sorted_ids)-1)
    return positions, (positions < len(sorted_ids)) & (sorted_ids[safe] == values)

annotation = feather.read_table(files['annotations']).to_pandas()
require_columns(annotation.columns, ['bodyId','status','superclass'], 'annotations')
neurons = annotation.loc[annotation.status == 'Traced'].sort_values('bodyId').set_index('bodyId')
assert neurons.index.is_unique and not neurons.index.hasnans
body_ids = neurons.index.to_numpy(np.int64)
n = len(body_ids)
CFG['n_neurons'] = n
assert n > 100_000, f'전체 Traced 데이터인지 확인하세요: {n}'
print('Actual full Traced neuron count:', n)
print(neurons['superclass'].value_counts(dropna=False).to_string())

def full_edge_batches():
    with pa.memory_map(str(files['edges']), 'r') as source:
        reader = pa.ipc.open_file(source)
        require_columns(reader.schema.names, ['body_pre','body_post','weight'], 'edges')
        for i in range(reader.num_record_batches):
            batch = reader.get_batch(i)
            s, sm = map_ids(body_ids, batch.column('body_pre').to_numpy(zero_copy_only=False))
            d, dm = map_ids(body_ids, batch.column('body_post').to_numpy(zero_copy_only=False))
            keep = sm & dm
            if keep.any():
                w = batch.column('weight').to_numpy(zero_copy_only=False)[keep]
                if not np.isfinite(w).all() or (w <= 0).any():
                    raise ValueError('Invalid count')
                yield s[keep].astype(np.int32), d[keep].astype(np.int32), w.astype(np.int64)

extract_spec = dict(version=3, selection='all_status_Traced_no_component_filter',
                    orientation='post_row_pre_column', n=n,
                    source_sha256={k:v['sha256'] for k,v in source_manifest.items()})
cache_key = hashlib.sha256(json.dumps(extract_spec, sort_keys=True).encode()).hexdigest()[:16]
full_cache = CACHE / f'full_{cache_key}'
full_cache.mkdir(exist_ok=True)
manifest_path = full_cache / 'complete.json'
if not manifest_path.exists():
    degree = np.zeros(n, np.int64)
    for s, d, w in tqdm(full_edge_batches(), desc='Pass 1: full degrees'):
        degree += np.bincount(d, minlength=n)
    raw_edges = int(degree.sum())
    if raw_edges == 0:
        raise ValueError('No full-neuron edges')
    required_disk = raw_edges*24 + 512*1024**2
    if shutil.disk_usage(full_cache).free < required_disk:
        raise RuntimeError(f'CSR 구축 디스크가 부족합니다. 약 {required_disk/2**30:.1f} GiB 여유가 필요합니다.')
    crow = np.concatenate(([0], np.cumsum(degree)))
    col = np.lib.format.open_memmap(full_cache/'src.npy', mode='w+', dtype=np.int32, shape=(raw_edges,))
    count = np.lib.format.open_memmap(full_cache/'count.npy', mode='w+', dtype=np.int64, shape=(raw_edges,))
    cursor = crow[:-1].copy()
    for s, d, w in tqdm(full_edge_batches(), desc='Pass 2: write full CSR'):
        order = np.argsort(d, kind='stable')
        sd = d[order]
        starts = np.r_[0, np.flatnonzero(sd[1:] != sd[:-1])+1]
        sizes = np.diff(np.r_[starts, len(sd)])
        local = np.arange(len(sd))-np.repeat(starts, sizes)
        positions = cursor[sd]+local
        col[positions], count[positions] = s[order], w[order]
        cursor += np.bincount(d, minlength=n)
    assert np.array_equal(cursor, crow[1:])
    col.flush()
    count.flush()
    full = sparse.csr_matrix((count, col, crow), shape=(n,n), copy=False)
    full.sum_duplicates()
    full.sort_indices()
    nnz = int(full.nnz)
    # sum_duplicates may shrink arrays/reallocate; persist the canonical prefix explicitly.
    col[:nnz], count[:nnz] = full.indices, full.data
    col.flush()
    count.flush()
    np.save(full_cache/'crow.npy', full.indptr.astype(np.int64))
    np.save(full_cache/'body_ids.npy', body_ids)
    manifest_path.write_text(json.dumps(dict(spec=extract_spec, raw_edges=raw_edges, nnz=nnz)), encoding='utf-8')
    del full, col, count, cursor, degree, crow
    gc.collect()
full_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert full_manifest['spec'] == extract_spec
crow = np.load(full_cache/'crow.npy', mmap_mode='r')
src = np.load(full_cache/'src.npy', mmap_mode='r')[:full_manifest['nnz']]
counts = np.load(full_cache/'count.npy', mmap_mode='r')[:full_manifest['nnz']]
assert int(crow[-1]) == len(src) == len(counts)
assert np.array_equal(body_ids, np.load(full_cache/'body_ids.npy'))
graph_data = dict(body_ids=body_ids, crow=crow, src=src, counts=counts)
print(dict(neurons=n, edges=len(src), retained='all Traced-to-Traced pairs'))


Actual full Traced neuron count: 165122
superclass
ol_intrinsic             89390
cb_intrinsic             32160
vnc_intrinsic            13151
visual_projection         9201
vnc_sensory               6365
cb_sensory                4868
ol_sensory                4114
ascending_neuron          1846
descending_neuron         1314
vnc_motor                  708
visual_centrifugal         563
sensory_ascending          537
None                       516
cb_motor                   107
vnc_efferent                94
cb_endocrine                72
ENS                         47
vnc_endocrine               22
sensory_descending          12
vnc_sensory_tbc              9
efferent_ascending           8
cb_sensory_tbc               5
efferent_descending          4
cb_efferent                  4
visual_projection_tbc        2
sensory_ascending_tbc        2
vnc_tbc                      1
{'neurons': 165122, 'edges': 25563197, 'retained': 'all Traced-to-Traced pairs'}


## 4. 실제 sensory/motor 포트와 세포 유형

전체 Traced 집합을 유지한다. 경로가 없는 뉴런도 삭제하지 않는다.
신경전달물질은 ground truth 우선, 없으면 confidence ≥ 0.8의 예측을 사용한다.
ACh 양수 / GABA 음수는 이 모델의 근사다. 수용체 정보가 없는 다른 전달물질은 부호를 학습한다.
도파민 등을 이름만으로 보상 학습 규칙으로 해석하지 않는다.
`type` annotation이 있는 뉴런은 같은 유형끼리 동역학 매개변수를 공유하고,
미분류 뉴런은 `class`, `superclass` 순으로 묶는다. 이 fallback도 저장한다.

In [8]:
KNOWN_NT = {'acetylcholine', 'gaba', 'glutamate', 'histamine', 'dopamine', 'serotonin', 'octopamine'}

def attach_nt(body_ids):
    selected_parts = []
    wanted = np.sort(body_ids)
    with pa.memory_map(str(files['nt']), 'r') as source:
        reader = pa.ipc.open_file(source)
        require_columns(reader.schema.names, ['body', 'ground_truth', 'predicted_nt',
                                              'predicted_nt_confidence', 'consensus_nt'], 'neurotransmitters')
        for i in range(reader.num_record_batches):
            batch = reader.get_batch(i)
            _, keep = map_ids(wanted, batch.column('body').to_numpy(zero_copy_only=False))
            if keep.any():
                selected_parts.append(batch.filter(pa.array(keep)).to_pandas())
    if not selected_parts:
        raise RuntimeError('선택된 body와 전달물질 테이블의 join이 비어 있습니다.')
    table = pd.concat(selected_parts, ignore_index=True)
    if table['body'].duplicated().any():
        raise ValueError('Duplicate NT body IDs: explicit aggregation is required')
    table = table.set_index('body').reindex(body_ids)
    truth = table['ground_truth'].fillna('').str.strip().str.lower()
    predicted = table['predicted_nt'].fillna('').str.strip().str.lower()
    confidence = pd.to_numeric(table['predicted_nt_confidence'], errors='coerce')
    trusted = truth.isin(KNOWN_NT)
    confident = predicted.isin(KNOWN_NT) & confidence.ge(CFG['nt_confidence']) & confidence.le(1)
    effective = np.where(trusted, truth, np.where(confident, predicted, 'unknown'))
    origin = np.where(trusted, 'ground_truth', np.where(confident, 'predicted_nt', 'unknown'))
    table['effective_nt'] = effective
    table['effective_source'] = origin
    table.index.name = 'body'
    return table


SENSORY_CLASSES = ['cb_sensory','ol_sensory','vnc_sensory','sensory_ascending','sensory_descending']
MOTOR_CLASSES = ['cb_motor','vnc_motor']
input_nodes = np.flatnonzero(neurons.superclass.isin(SENSORY_CLASSES)).astype(np.int64)
output_nodes = np.flatnonzero(neurons.superclass.isin(MOTOR_CLASSES)).astype(np.int64)
assert len(input_nodes) and len(output_nodes)
assert len(np.intersect1d(input_nodes, output_nodes)) == 0
nt_table = attach_nt(body_ids)
node_sign = nt_table.effective_nt.map({'acetylcholine':1.0,'gaba':-1.0}).fillna(0).to_numpy(np.float32)
if not CFG['use_nt_signs']:
    node_sign[:] = 0
edge_sign = node_sign[src]
from scipy.sparse.csgraph import dijkstra
# Multi-source min_only returns a length-N vector, never an inputs×N dense distance matrix.
adj_post_pre = sparse.csr_matrix((np.ones(len(src), np.uint8), src, crow), shape=(n,n), copy=False)
adj_pre_post = adj_post_pre.T.tocsr()
from_sensory = dijkstra(adj_pre_post, directed=True, indices=input_nodes, min_only=True, unweighted=True)
to_motor = dijkstra(adj_post_pre, directed=True, indices=output_nodes, min_only=True, unweighted=True)
graph_stats = dict(neurons=n, edges=len(src), sensory_neurons=len(input_nodes), motor_neurons=len(output_nodes),
                   reachable_from_sensory=int(np.isfinite(from_sensory).sum()),
                   can_reach_motor=int(np.isfinite(to_motor).sum()),
                   sensory_to_motor_path_neurons=int((np.isfinite(from_sensory)&np.isfinite(to_motor)).sum()),
                   reachable_motors=int(np.isfinite(from_sensory[output_nodes]).sum()),
                   positive_edges=int((edge_sign>0).sum()), negative_edges=int((edge_sign<0).sum()),
                   unconstrained_edges=int((edge_sign==0).sum()))
assert graph_stats['reachable_motors'] > 0, 'No biological sensory→motor path; annotations/direction를 확인하세요.'
print(graph_stats)
print(nt_table.groupby(['effective_source','effective_nt']).size().to_string())
neurons[['superclass','class']].assign(is_input=neurons.index.isin(body_ids[input_nodes]),
               is_output=neurons.index.isin(body_ids[output_nodes])).to_csv(RUN_DIR/'neuron_ports.csv')
nt_table.to_csv(RUN_DIR/'neurotransmitters.csv')
del adj_post_pre, adj_pre_post, annotation
gc.collect()


def dynamics_types(frame):
    labels = pd.Series('', index=frame.index, dtype=object)
    for column in ['type', 'class', 'superclass']:
        if column not in frame:
            continue
        values = frame[column].fillna('').astype(str).str.strip()
        valid = ~values.str.lower().isin(['', 'none', 'nan', 'unknown', 'tbc', 'untyped'])
        use = labels.eq('') & valid
        labels.loc[use] = column + ':' + values.loc[use]
    labels = labels.mask(labels.eq(''), 'unknown')
    codes, names = pd.factorize(labels, sort=True)
    return codes.astype(np.int64), list(names)

type_ids, type_names = dynamics_types(neurons)
print('Dynamics groups:', len(type_names), 'type-annotated neurons:',
      sum(type_names[i].startswith('type:') for i in type_ids))

{'neurons': 165122, 'edges': 25563197, 'sensory_neurons': 15896, 'motor_neurons': 815, 'reachable_from_sensory': 164699, 'can_reach_motor': 163553, 'sensory_to_motor_path_neurons': 163531, 'reachable_motors': 815, 'positive_edges': 13413675, 'negative_edges': 4260040, 'unconstrained_edges': 7889482}
effective_source  effective_nt 
ground_truth      acetylcholine    52778
                  dopamine           380
                  gaba             13149
                  glutamate        14395
                  histamine         2699
                  octopamine          51
                  serotonin           44
predicted_nt      acetylcholine    43006
                  dopamine           230
                  gaba              6089
                  glutamate         7515
                  histamine          570
                  octopamine           9
                  serotonin           15
unknown           unknown          24192
Dynamics groups: 11779 type-annotated neurons: 16251

## 5. 유형별 동역학과 내부 상태

각 tick에서 이전 `s`가 연결망을 통해 전류를 전달하고, 막 상태가 이를 적분한다.
`r = relu(tanh(v − threshold))`는 0~1의 신호 방출량이다.
`s`는 빠르게, `a`는 느리게 `r`을 추적하며 `a`는 막 입력을 억제한다.
이는 스파이크를 직접 구현하지 않는 연속 신호 모델이다.

유형별 시간 상수는 동일한 초기값에서 시작해 개별적으로 학습한다. 초기 상태는 모두 0이다.
기존 시냅스 수의 log 정규화 prior를 사용하고, 알려진 부호의 연결은 양의 배율로만 조절한다.
임의 부호 연결은 bounded tanh로 조절한다. 이 가중치는 실제 전도도 측정값이 아니다.

In [11]:
BOS, EOS, VOCAB = 256, 257, 258

def encode(text):
    return [BOS] + list(text.encode('utf-8')) + [EOS]

def decode(ids):
    return bytes(int(i) for i in ids if 0 <= int(i) < 256).decode('utf-8', errors='replace')

class SparseEdges(torch.autograd.Function):
    @staticmethod
    def forward(ctx, h, weights, rowptr, col, dst, block):
        matrix = torch.sparse_csr_tensor(rowptr, col, weights, size=(h.shape[1], h.shape[1]),
                                         device=h.device, dtype=h.dtype)
        ctx.save_for_backward(h, weights, rowptr, col, dst)
        ctx.block = block
        return torch.sparse.mm(matrix, h.T.contiguous()).T.contiguous()

    @staticmethod
    def backward(ctx, grad_output):
        h, weights, rowptr, col, dst = ctx.saved_tensors
        grad_h = torch.zeros_like(h) if ctx.needs_input_grad[0] else None
        grad_w = torch.empty_like(weights) if ctx.needs_input_grad[1] else None
        for lo in range(0, col.numel(), ctx.block):
            hi = min(col.numel(), lo+ctx.block)
            sources, targets = col[lo:hi], dst[lo:hi]
            g = grad_output.index_select(1, targets)
            if grad_w is not None:
                grad_w[lo:hi] = (g*h.index_select(1, sources)).sum(0)
            if grad_h is not None:
                grad_h.index_add_(1, sources, g*weights[lo:hi])
        return grad_h, grad_w, None, None, None, None

class ConnectomeLM(nn.Module):
    def __init__(self, cfg, rowptr, src, counts, edge_sign, inputs, outputs, type_ids, device):
        super().__init__()
        self.n, self.k = cfg['n_neurons'], cfg['brain_steps']
        self.block, self.checkpoint_tokens = cfg['edge_block'], cfg['checkpoint_tokens']
        for name, value in [('rowptr', rowptr), ('src', src), ('inputs', inputs),
                            ('outputs', outputs), ('type_ids', type_ids)]:
            self.register_buffer(name, torch.tensor(np.asarray(value), dtype=torch.long,
                                                   device=device), persistent=False)
        assert len(type_ids) == self.n and min(type_ids) >= 0
        self.register_buffer('dst', torch.repeat_interleave(
            torch.arange(self.n, device=device), self.rowptr.diff()), persistent=False)
        sign = torch.tensor(np.asarray(edge_sign), dtype=torch.float32, device=device)
        self.register_buffer('edge_sign', sign, persistent=False)
        c = torch.tensor(np.asarray(counts), dtype=torch.float32, device=device).log1p()
        incoming = torch.zeros(self.n, device=device).index_add_(0, self.dst, c)
        self.register_buffer('prior', c / incoming[self.dst].clamp_min(1e-8), persistent=False)
        raw = torch.where(sign != 0, torch.full_like(sign, math.log(0.2 / 0.8)),
                          torch.where(torch.rand_like(sign) < 0.5, -0.2554128, 0.2554128))
        self.theta = nn.Parameter(raw)
        ntypes = int(max(type_ids)) + 1
        low, high, initial = [torch.tensor(cfg[key], device=device)[:, None]
                              for key in ['tau_min', 'tau_max', 'tau_initial']]
        assert torch.all((low > 0) & (initial > low) & (initial < high))
        self.register_buffer('tau_low', low)
        self.register_buffer('tau_high', high)
        fraction = (initial - low) / (high - low)
        self.raw_tau = nn.Parameter(torch.logit(fraction).expand(3, ntypes).clone())
        self.threshold = nn.Parameter(torch.zeros(ntypes, device=device))
        self.raw_adaptation = nn.Parameter(torch.full((ntypes,), math.log(0.1 / 0.9), device=device))
        d = cfg['embedding_dim']
        self.embedding = nn.Embedding(VOCAB, d, device=device)
        self.input_adapter = nn.Linear(d, len(inputs), bias=False, device=device)
        self.motor_norm = nn.LayerNorm(len(outputs), eps=1e-6, elementwise_affine=False, device=device)
        self.output_adapter = nn.Linear(len(outputs), d, bias=False, device=device)

    def effective_weights(self):
        magnitude = 0.25 + 3.75 * self.theta.sigmoid()
        signed = torch.where(self.edge_sign != 0, self.edge_sign * magnitude,
                             4 * self.theta.tanh())
        return self.prior * signed

    def dynamics(self):
        tau = self.tau_low + (self.tau_high - self.tau_low) * self.raw_tau.sigmoid()
        alpha = -torch.expm1(-1 / tau[:, self.type_ids])
        threshold = 0.5 * self.threshold.tanh()[self.type_ids]
        adaptation = self.raw_adaptation.sigmoid()[self.type_ids]
        return alpha, threshold, adaptation

    def initial_state(self, batch):
        return self.embedding.weight.new_zeros(3, batch, self.n)

    def advance(self, token, state, weights, alpha, threshold, adaptation):
        v, signal, slow = state.unbind(0)
        value = self.input_adapter(self.embedding(token))
        drive = torch.zeros_like(v).index_add(1, self.inputs, value)
        for _ in range(self.k):
            message = SparseEdges.apply(signal, weights, self.rowptr, self.src, self.dst, self.block)
            v = v + alpha[0] * (-v + message + drive - adaptation * slow)
            rate = torch.relu(torch.tanh(v - threshold))
            signal = signal + alpha[1] * (rate - signal)
            slow = slow + alpha[2] * (rate - slow)
        return torch.stack((v, signal, slow))

    def forward(self, tokens, state=None, active_mask=None):
        if tokens.ndim != 2 or tokens.shape[1] == 0:
            raise ValueError('Nonempty tokens [B,T] required')
        state = self.initial_state(tokens.shape[0]) if state is None else state
        if state.shape != (3, tokens.shape[0], self.n):
            raise ValueError('State must be [3,B,N]: membrane, synaptic signal, adaptation')
        if active_mask is not None and (active_mask.shape != tokens.shape or active_mask.dtype != torch.bool):
            raise ValueError('active_mask must be bool [B,T]')
        weights = self.effective_weights()
        alpha, threshold, adaptation = self.dynamics()
        logits = []
        for t in range(tokens.shape[1]):
            previous = state
            args = (tokens[:, t], state, weights, alpha, threshold, adaptation)
            if self.training and self.checkpoint_tokens and torch.is_grad_enabled():
                state = checkpoint(self.advance, *args, use_reentrant=False, preserve_rng_state=False)
            else:
                state = self.advance(*args)
            if active_mask is not None:
                state = torch.where(active_mask[:, t][None, :, None], state, previous)
            motor = self.motor_norm(state[1].index_select(1, self.outputs))
            logits.append(F.linear(self.output_adapter(motor), self.embedding.weight))
        return torch.stack(logits, dim=1), state

def make_model():
    torch.manual_seed(CFG['seed'])
    torch.cuda.manual_seed_all(CFG['seed'])
    return ConnectomeLM(CFG, crow, src, counts, edge_sign, input_nodes, output_nodes, type_ids, device)

# ponytail: single compartment per neuron; add anatomical compartments only when needed.

## 6. 기본 검증과 VRAM 자동 배치 탐색

작은 그래프의 수치 검증 후 **전체 MaleCNS + 실제 64-token backward/Adam**으로 배치를 탐색한다.
표의 `peak_reserved_GiB`는 allocator 여유분을 포함한다. 메모리 부족 시 해당 배치만 실패 처리한다.
배치 1조차 예산에 들어오지 않으면 학습을 시작하지 않는다. 뉴런/연결은 줄이지 않는다.
탐색은 메모리 적합성을 기준으로 하며, 가장 빠른 배치나 실제 대화 능력을 보장하지 않는다.

In [12]:
CFG['batch_size'] = 2

## 7. 실제 다중 배치 학습

비슷한 prompt/답변 길이끼리 배치를 구성해 padding 낭비를 줄인다.
각 대화의 prompt 앞부분은 독립적으로 warmup하고, 마지막 최대 32 prompt bytes는 답변과 함께 역전파한다.
padding 동안 해당 행의 신경 상태는 멈추고 loss에서도 제외한다. 배치가 바뀔 때 모든 행의 상태를 초기화한다.
loss는 padding을 제외한 전체 답변 byte 수로 나눈다. TBPTT는 각 대화별 64단계로 유지한다.
batch update 수와 `response_tokens_per_second`를 함께 출력한다.
checkpoint는 추론/추가 학습용 가중치이며 optimizer/data cursor의 정확한 재개는 제공하지 않는다.

In [13]:
IGNORE_INDEX = -100
assert (BOS, EOS) == (256, 257)

def response_mask(start, length, prompt_tokens):
    # x[k]가 예측하는 y[k]의 원래 sequence 위치는 start+k+1이다.
    return np.arange(start+1, start+length+1) >= prompt_tokens

def chunk_spans(length, prompt_tokens):
    warmup = max(0, prompt_tokens - 1 - CFG['prompt_grad_tokens'])
    for start in range(0, warmup, CFG['tbptt']):
        yield start, min(CFG['tbptt'], warmup - start)
    for start in range(warmup, length - 1, CFG['tbptt']):
        yield start, min(CFG['tbptt'], length - 1 - start)

def batched_chunks(indices, max_response_tokens=None):
    rows = [pairs[i] for i in indices]
    warmups = [max(0, row['prompt_tokens'] - 1 - CFG['prompt_grad_tokens']) for row in rows]
    stops = [row['length'] if max_response_tokens is None else
             min(row['length'], row['prompt_tokens'] + max_response_tokens) for row in rows]
    # First consume every row's warmup, freezing shorter rows at their own boundary.
    for warmup_phase in (True, False):
        starts = [0] * len(rows) if warmup_phase else warmups
        lengths = warmups if warmup_phase else [stop - 1 - start for stop, start in zip(stops, starts)]
        for offset in range(0, max(lengths), CFG['tbptt']):
            width = min(CFG['tbptt'], max(lengths) - offset)
            x = np.zeros((len(rows), width), dtype=np.int64)
            y = np.full_like(x, IGNORE_INDEX)
            active = np.zeros(x.shape, dtype=bool)
            counts = np.zeros(len(rows), dtype=np.int64)
            for lane, row in enumerate(rows):
                take = min(width, max(0, lengths[lane] - offset))
                if take == 0:
                    continue
                start = starts[lane] + offset
                begin = row['offset'] + start
                values = tokens_mm[begin:begin+take+1]
                x[lane, :take] = values[:-1]
                active[lane, :take] = True
                if not warmup_phase:
                    mask = response_mask(start, take, row['prompt_tokens'])
                    # Copy uint16 tokens to signed int64 BEFORE inserting -100.
                    y[lane, :take] = values[1:]
                    y[lane, :take][~mask] = IGNORE_INDEX
                    counts[lane] = int(mask.sum())
            # Reject invalid IDs on CPU before a CUDA assert poisons the kernel.
            invalid_x = (x < 0) | (x >= VOCAB)
            invalid_y = (y != IGNORE_INDEX) & ((y < 0) | (y >= VOCAB))
            if invalid_x.any() or invalid_y.any():
                raise ValueError(f'Invalid token IDs in pairs {indices}: '
                                 f'inputs={x[invalid_x][:8].tolist()}, labels={y[invalid_y][:8].tolist()}')
            yield (torch.from_numpy(x).to(device), torch.from_numpy(y).to(device),
                   torch.from_numpy(active).to(device), counts)

def epoch_batches(epoch):
    rng = random.Random(CFG['seed'] + epoch)
    order = train_ids.copy()
    rng.shuffle(order)
    batch = CFG['batch_size']
    groups = []
    def length_key(index):
        row = pairs[index]
        warmup = max(0, row['prompt_tokens'] - 1 - CFG['prompt_grad_tokens'])
        return (warmup // CFG['tbptt'], (row['length'] - warmup) // CFG['tbptt'])
    for start in range(0, len(order), batch * 32):
        pool = sorted(order[start:start+batch*32], key=length_key)
        groups.extend(pool[i:i+batch] for i in range(0, len(pool), batch))
    rng.shuffle(groups)
    return groups

def batch_updates(indices):
    return max(math.ceil((pairs[i]['length'] - 1 - max(
        0, pairs[i]['prompt_tokens'] - 1 - CFG['prompt_grad_tokens'])) / CFG['tbptt']) for i in indices)

def check_response_format():
    fixture = dict(utterance=[
        dict(speaker_id='A', form='질문 하나'), dict(speaker_id='A', form='질문 둘'),
        dict(speaker_id='B', form='답변 하나'), dict(speaker_id='A', form='다음 질문'),
        dict(speaker_id='B', form='다음 답변')])
    # text_field 설정과 무관하게 동일 구조를 검사한다.
    for utterance in fixture['utterance']:
        utterance[DATA_CFG['text_field']] = utterance['form']
    turns, reason = dialogue_turns(fixture, 'fixture')
    assert reason is None and turns == [(1, '질문 하나 질문 둘'), (2, '답변 하나'), (1, '다음 질문'), (2, '다음 답변')]
    assert dialogue_turns(dict(utterance=fixture['utterance'][:2]), 'monologue')[1] == 'single_speaker'
    encoded, boundary = encode_pair('질문', '답변')
    actual_targets = []
    for start, length in chunk_spans(len(encoded), boundary):
        mask = response_mask(start, length, boundary)
        actual_targets.extend(encoded[start+1:start+length+1][mask].tolist())
    assert actual_targets == list('답변'.encode('utf-8')) + [EOS]
    assert not response_mask(0, boundary-1, boundary).any()
    assert response_mask(boundary-1, 1, boundary)[0]  # 첫 답변 byte가 loss에 포함됨
    print('PASS: speaker mapping, turn merge, monologue exclusion, response-only mask + EOS')

check_response_format()
eval_ids = sorted(val_ids, key=lambda i: pairs[i]['sha256'])[:CFG['eval_max_pairs']]


@torch.no_grad()
def evaluate(m):
    was_training = m.training
    m.eval()
    loss_sum, correct, total = 0., 0, 0
    try:
        for start in range(0, len(eval_ids), CFG['batch_size']):
            state = None
            for x, labels, active, counts in batched_chunks(eval_ids[start:start+CFG['batch_size']], CFG['eval_response_tokens']):
                logits, state = m(x, state, active_mask=active)
                supervised = int(counts.sum())
                if supervised:
                    loss_sum += F.cross_entropy(logits.reshape(-1, VOCAB), labels.reshape(-1),
                                                ignore_index=IGNORE_INDEX, reduction='sum').item()
                    correct += int(((logits.argmax(-1) == labels) & (labels != IGNORE_INDEX)).sum())
                    total += supervised
    finally:
        m.train(was_training)
    mean = loss_sum / total
    if not math.isfinite(mean):
        raise RuntimeError('Nonfinite validation loss')
    return dict(response_loss=mean, response_perplexity=math.exp(mean) if mean < 700 else None,
                response_accuracy=correct/total, response_tokens=total, pairs=len(eval_ids),
                full_validation=total == data_stats['validation_response_tokens'])

def to_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {key: to_cpu(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_cpu(item) for item in value]
    if isinstance(value, tuple):
        return tuple(to_cpu(item) for item in value)
    return value


# Immutable full graph is exported once, avoiding duplicate graph copies in each checkpoint.
graph_export = RUN_DIR/'graph'
graph_export.mkdir(exist_ok=True)
for filename in ['src.npy','count.npy','crow.npy','body_ids.npy','complete.json']:
    if not (graph_export/filename).exists():
        shutil.copyfile(full_cache/filename,graph_export/filename)
np.save(graph_export/'edge_sign.npy',edge_sign)
np.save(graph_export/'inputs.npy',input_nodes)
np.save(graph_export/'outputs.npy',output_nodes)
np.save(graph_export/'type_ids.npy',type_ids)
(graph_export/'type_names.json').write_text(json.dumps(type_names, ensure_ascii=False), encoding='utf-8')

def save_checkpoint(name,m,optimizer,updates,validation):
    # Parameter-only full-graph checkpoint: optimizer/data cursor are not saved.
    payload = dict(architecture='malecns_dynamics_v1', config=CFG,model_state=to_cpu(m.state_dict()),graph_directory='graph',
                   extract_spec=extract_spec,sources=source_manifest,data_spec=data_spec,
                   data_config=DATA_CFG,updates=updates,validation=validation)
    local = CACHE/name
    torch.save(payload,local)
    temporary = RUN_DIR/(name+'.partial')
    shutil.copyfile(local,temporary)
    temporary.replace(RUN_DIR/name)
    del payload

def save_history(rows):
    (RUN_DIR / 'history.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')

PASS: speaker mapping, turn merge, monologue exclusion, response-only mask + EOS


In [14]:
model = make_model()  # 재실행은 새 학습이다. 자동 탐색 모델은 사용하지 않는다.
optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'], foreach=False)
initial = dict(evaluate(model), evaluated_at_update=0)
best_loss, last_validation = initial['response_loss'], initial
history = [dict(update=0, batch_size=CFG['batch_size'], validation=initial)]
save_checkpoint('best.pt', model, optimizer, 0, initial)
save_history(history)
updates, seen_response_tokens, processed_tokens, completed_epochs = 0, 0, 0, 0
pairs_started, pairs_completed = 0, 0
training_seconds, window_loss, window_tokens = 0.0, 0.0, 0
planned_updates = sum(batch_updates(group) for epoch in range(CFG['epochs']) for group in epoch_batches(epoch))
if CFG['train_updates'] is not None:
    planned_updates = min(planned_updates, CFG['train_updates'])
torch.cuda.reset_peak_memory_stats()
wall_start = time.perf_counter()
with tqdm(total=planned_updates, desc=f"Train batch={CFG['batch_size']}") as progress:
    for epoch in range(CFG['epochs']):
        finished_in_epoch = 0
        for indices in epoch_batches(epoch):
            if updates >= planned_updates:
                break
            state = None
            response_seen = np.zeros(len(indices), dtype=np.int64)
            pairs_started += len(indices)
            for x, labels, active, counts in batched_chunks(indices):
                if updates >= planned_updates:
                    break
                model.train()
                supervised = int(counts.sum())
                torch.cuda.synchronize()
                tick = time.perf_counter()
                if supervised == 0:
                    with torch.no_grad():
                        _, state = model(x, state, active_mask=active)
                    state = state.detach()
                    torch.cuda.synchronize()
                    training_seconds += time.perf_counter() - tick
                    processed_tokens += x.numel()
                    continue
                optimizer.zero_grad(set_to_none=True)
                logits, next_state = model(x, state, active_mask=active)
                loss = F.cross_entropy(logits.reshape(-1, VOCAB), labels.reshape(-1), ignore_index=IGNORE_INDEX)
                if not torch.isfinite(loss):
                    raise RuntimeError(f'Nonfinite loss at batch update {updates+1}')
                loss.backward()
                grad_norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
                optimizer.step()
                state = next_state.detach()
                loss_value, grad_value = loss.item(), float(grad_norm)
                # Free each chunk's graph/gradients before evaluation or the next batch.
                del logits, next_state, loss, grad_norm
                optimizer.zero_grad(set_to_none=True)
                torch.cuda.synchronize()
                training_seconds += time.perf_counter() - tick
                updates += 1
                processed_tokens += x.numel()
                seen_response_tokens += supervised
                response_seen += counts
                window_loss += loss_value * supervised
                window_tokens += supervised
                progress.update(1)
                if updates == 1 or updates % CFG['log_every'] == 0:
                    row = dict(update=updates, epoch=epoch+1, actual_batch=len(indices),
                               response_tokens=seen_response_tokens, train_response_loss=window_loss/window_tokens,
                               grad_norm=grad_value, response_tokens_per_second=seen_response_tokens/training_seconds,
                               peak_allocated_GiB=torch.cuda.max_memory_allocated()/2**30,
                               peak_reserved_GiB=torch.cuda.max_memory_reserved()/2**30)
                    history.append(row)
                    tqdm.write(json.dumps(row))
                    window_loss, window_tokens = 0.0, 0
                if updates % CFG['eval_every'] == 0:
                    last_validation = dict(evaluate(model), evaluated_at_update=updates)
                    history.append(dict(update=updates, validation=last_validation))
                    tqdm.write(json.dumps(dict(update=updates, validation=last_validation)))
                    if last_validation['response_loss'] < best_loss:
                        best_loss = last_validation['response_loss']
                        save_checkpoint('best.pt', model, optimizer, updates, last_validation)
                if updates % CFG['save_every'] == 0:
                    save_checkpoint('latest.pt', model, optimizer, updates, last_validation)
                    save_history(history)
            completed = sum(int(seen == pairs[index]['length'] - pairs[index]['prompt_tokens'])
                            for index, seen in zip(indices, response_seen))
            pairs_completed += completed
            finished_in_epoch += completed
            del state
        if finished_in_epoch == len(train_ids):
            completed_epochs += 1
        if updates >= planned_updates:
            break
final = dict(evaluate(model), evaluated_at_update=updates)
history.append(dict(update=updates, validation=final, final=True))
if final['response_loss'] < best_loss:
    best_loss = final['response_loss']
    save_checkpoint('best.pt', model, optimizer, updates, final)
save_checkpoint('latest.pt', model, optimizer, updates, final)
save_history(history)
torch.cuda.synchronize()
report = dict(batch_size=CFG['batch_size'], initial_validation=initial, final_validation=final,
              best_response_loss=best_loss, updates=updates, train_response_tokens=seen_response_tokens,
              processed_model_tokens_including_padding=processed_tokens,
              pairs_started=pairs_started, pairs_completed=pairs_completed, completed_epochs=completed_epochs,
              stop_reason='epochs_completed' if completed_epochs == CFG['epochs'] else 'update_limit',
              training_seconds=training_seconds, response_tokens_per_second=seen_response_tokens/training_seconds,
              wall_seconds=time.perf_counter()-wall_start,
              peak_allocated_MiB=torch.cuda.max_memory_allocated()/2**20,
              peak_reserved_MiB=torch.cuda.max_memory_reserved()/2**20)
print(json.dumps(report, ensure_ascii=False, indent=2))

Train batch=2:   0%|          | 0/1000 [00:00<?, ?it/s]

{"update": 1, "epoch": 1, "actual_batch": 2, "response_tokens": 64, "train_response_loss": 9.153958320617676, "grad_norm": 170.40185546875, "response_tokens_per_second": 4.83818148212442, "peak_allocated_GiB": 1.4845690727233887, "peak_reserved_GiB": 1.55859375}
{"update": 50, "epoch": 1, "actual_batch": 2, "response_tokens": 3426, "train_response_loss": 5.937832785390233, "grad_norm": 34.08218002319336, "response_tokens_per_second": 7.802657977309317, "peak_allocated_GiB": 1.68341064453125, "peak_reserved_GiB": 1.751953125}
{"update": 100, "epoch": 1, "actual_batch": 2, "response_tokens": 6841, "train_response_loss": 4.377540245098041, "grad_norm": 27.211851119995117, "response_tokens_per_second": 8.031695779145261, "peak_allocated_GiB": 1.6834125518798828, "peak_reserved_GiB": 1.751953125}
{"update": 150, "epoch": 1, "actual_batch": 2, "response_tokens": 10992, "train_response_loss": 3.800191834414617, "grad_norm": 49.401947021484375, "response_tokens_per_second": 8.430731686308153, 

## 8. 생성과 가중치 복원

prompt를 한 번 읽고, 그 다음에는 생성한 byte와 뉴런 상태만 전달한다.
UTF-8의 유효한 다음 byte만 허용한다. temperature=0은 greedy, 양수는 sampling이다.
이 제약은 인코딩만 보장하며 문장 품질을 보장하지 않는다.
실행 예제는 학습 결과를 그대로 출력한다. best checkpoint는 모델을 하나만 유지하면서 복원한다.

In [15]:
import codecs

def inference_inputs(question):
    question = ' '.join(question.split())
    if not question:
        raise ValueError('Nonempty question required')
    return [BOS] + list(('화자1: ' + question + '\n화자2: ').encode('utf-8'))

@torch.no_grad()
def prefill_question(m, question):
    prefix = inference_inputs(question)
    state = None
    for start in range(0, len(prefix), CFG['tbptt']):
        ids = torch.tensor([prefix[start:start+CFG['tbptt']]], device=m.embedding.weight.device)
        logits, state = m(ids, state)
    return logits, state

def valid_utf8_tokens(decoder):
    state = decoder.getstate()
    probe = codecs.getincrementaldecoder('utf-8')('strict')
    allowed = []
    for value in range(256):
        probe.setstate(state)
        try:
            probe.decode(bytes([value]), final=False)
            allowed.append(value)
        except UnicodeDecodeError:
            pass
    if not state[0]:
        allowed.append(EOS)
    return allowed

@torch.no_grad()
def reply(m, question, max_new_bytes=180, temperature=0.8):
    if max_new_bytes < 1 or temperature < 0:
        raise ValueError('max_new_bytes >= 1, temperature >= 0 required')
    was_training = m.training
    m.eval()
    decoder = codecs.getincrementaldecoder('utf-8')('strict')
    pieces = []
    try:
        logits, state = prefill_question(m, question)
        for _ in range(max_new_bytes):
            allowed = torch.tensor(valid_utf8_tokens(decoder), device=logits.device)
            scores = logits[0, -1].index_select(0, allowed)
            if not torch.isfinite(scores).all():
                raise RuntimeError('Nonfinite generation logits')
            selected = scores.argmax() if temperature == 0 else torch.multinomial(
                (scores / temperature).softmax(-1), 1)[0]
            token = int(allowed[selected])
            if token == EOS:
                break
            pieces.append(decoder.decode(bytes([token]), final=False))
            logits, state = m(torch.tensor([[token]], device=logits.device), state)
        # A partial UTF-8 character at the byte budget is omitted, never replaced with U+FFFD.
        return ''.join(pieces)
    finally:
        m.train(was_training)

def load_full_checkpoint(checkpoint_path):
    saved = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if saved.get('architecture') != 'malecns_dynamics_v1':
        raise ValueError('이 노트북의 dynamics checkpoint만 복원할 수 있습니다.')
    folder = Path(checkpoint_path).parent / saved['graph_directory']
    info = json.loads((folder/'complete.json').read_text())
    count = info['nnz']
    restored = ConnectomeLM(saved['config'], np.load(folder/'crow.npy', mmap_mode='r'),
        np.load(folder/'src.npy', mmap_mode='r')[:count],
        np.load(folder/'count.npy', mmap_mode='r')[:count],
        np.load(folder/'edge_sign.npy', mmap_mode='r'),
        np.load(folder/'inputs.npy'), np.load(folder/'outputs.npy'),
        np.load(folder/'type_ids.npy'), device)
    restored.load_state_dict(saved['model_state'])
    return restored.eval()

# Free Adam and current model before loading the best full graph model.
del optimizer, model
gc.collect()
torch.cuda.empty_cache()
model = load_full_checkpoint(RUN_DIR/'best.pt')
examples = [dict(input=q, output=reply(model, q)) for q in
            ['안녕하세요?', '좋아하는 음식은 뭐예요?', '기차는 어디에서 타나요?']]
print(json.dumps(examples, ensure_ascii=False, indent=2))
(RUN_DIR/'examples.json').write_text(json.dumps(examples, ensure_ascii=False, indent=2), encoding='utf-8')
(RUN_DIR/'result.json').write_text(json.dumps(dict(config=CFG, versions=VERSIONS,
    sources=source_manifest, data_config=DATA_CFG, data_spec=data_spec, data_stats=data_stats,
    graph=graph_stats, dynamics_groups=len(type_names), metrics=report),
    ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', RUN_DIR)

# print(reply(model, '오늘 기분이 어때요?', temperature=0))

[
  {
    "input": "안녕하세요?",
    "output": "엑응 그닠 한는 보는데단데 그래더라다 그래도 거 그래는."
  },
  {
    "input": "좋아하는 음식은 뭐예요?",
    "output": "마은 그래지 그래지. 그래서녌볋고 그런야?"
  },
  {
    "input": "기차는 어디에서 타나요?",
    "output": "됎듘는 그다 땜 됨더 그래없많을 해지만에 맘 먠는Ꟶ붋든 .반도 비다 나는 그단 녔라 큜다 되는 됬는데 그래 빐 혴니 근던 됊뗐 된 맠제"
  }
]
Saved: /kaggle/working/flylm_runs/20260908_014746_715747


In [17]:
from pathlib import Path
import shutil

# 검증 성능이 가장 좋았던 모델.
# 최종 학습본을 저장하려면 'latest.pt'로 변경하세요.
checkpoint_name = 'best.pt'

run_dir = Path(RUN_DIR)
checkpoint_path = run_dir / checkpoint_name
graph_dir = run_dir / 'graph'
export_dir = run_dir / 'flylm_inference'

if not checkpoint_path.is_file():
    raise FileNotFoundError(f'모델 파일이 없습니다: {checkpoint_path}')
if not graph_dir.is_dir():
    raise FileNotFoundError(f'뇌 연결망 폴더가 없습니다: {graph_dir}')

export_dir.mkdir(parents=True, exist_ok=True)

shutil.copyfile(checkpoint_path, export_dir / 'model.pt')
shutil.copytree(
    graph_dir,
    export_dir / 'graph',
    dirs_exist_ok=True,
)

zip_path = shutil.make_archive(
    str(export_dir),
    'zip',
    root_dir=run_dir,
    base_dir=export_dir.name,
)

print('저장 완료:', zip_path)
print('Kaggle Files에서 이 ZIP을 다운로드하세요.')

저장 완료: /kaggle/working/flylm_runs/20260908_014746_715747/flylm_inference.zip
Kaggle Files에서 이 ZIP을 다운로드하세요.


## 실행 결과 확인

- `Selected real training batch`에서 자동 선택된 배치를 확인한다.
- `batch_tuning.json`에는 실제 전체 그래프 배치별 peak VRAM과 실행 시간이 저장된다.
- 학습 로그의 `actual_batch`, `response_tokens_per_second`, `peak_reserved_GiB`를 확인한다.
- 메모리 목표는 최악의 64-token 배치 기준이다. 짧은 배치와 warmup에서는 사용량이 낮아질 수 있다.
- Kaggle `/kaggle/working/flylm_runs`에 graph, 가중치, 데이터 manifest, 지표, 생성 예제를 저장한다.
- 1,000 batch update는 완성된 대화를 보장하지 않는다. 이전 batch 1보다 학습량이 커졌다.